# P04A — Generator data sintesis algoritma

Menyiapkan empat data latihan ber-seed tetap untuk contoh regresi, klasifikasi, clustering dan anomali. Seluruh nilai/label berasal dari aturan simulasi, bukan pengukuran lapangan. Jalankan notebook ini lebih dahulu.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs

SEED = 41804
rng = np.random.default_rng(SEED)
out_dir = Path("data_bab4")
out_dir.mkdir(exist_ok=True)

## 1. Regresi sintesis kuat tekan beton
Rumus memperlihatkan pengaruh campuran dan umur dengan noise. Koefisien bukan kalibrasi material dan tidak dapat dipakai untuk desain campuran.

In [2]:
n = 1200
semen = rng.uniform(220, 480, n)
air = rng.uniform(135, 245, n)
fly_ash = rng.uniform(0, 120, n)
superplasticizer = rng.uniform(0, 18, n)
agregat_kasar = rng.uniform(850, 1150, n)
agregat_halus = rng.uniform(600, 900, n)
umur = rng.choice([3, 7, 14, 28, 56, 90], n)
rasio = air / semen
kuat_tekan = (8 + 0.055*semen - 0.085*air + 0.025*fly_ash
              + 0.55*superplasticizer + 5.2*np.log(umur)
              + rng.normal(0, 3.0, n))
beton = pd.DataFrame({
    "id_spesimen": [f"B{i:04d}" for i in range(1, n+1)],
    "semen_kg_m3": semen.round(1), "air_kg_m3": air.round(1),
    "rasio_air_semen": rasio.round(3), "fly_ash_kg_m3": fly_ash.round(1),
    "superplasticizer_kg_m3": superplasticizer.round(2),
    "agregat_kasar_kg_m3": agregat_kasar.round(1),
    "agregat_halus_kg_m3": agregat_halus.round(1), "umur_hari": umur,
    "kuat_tekan_MPa": kuat_tekan.round(2)})
beton.to_csv(out_dir / "data_sintesis_kuat_tekan.csv", index=False)
beton.head()

## 2. Klasifikasi sintesis prioritas inspeksi jembatan
Label simulasi merupakan aturan latihan, bukan rekomendasi manajemen aset.

In [3]:
m = 600
umur_jembatan = rng.integers(1, 101, m)
lalin_berat = rng.integers(100, 12001, m)
risiko_gerusan = rng.uniform(0, 1, m)
indeks_deteriorasi = rng.uniform(0, 1, m)
paparan_banjir = rng.uniform(0, 1, m)
skor = (.004*umur_jembatan + .0001*lalin_berat
        + .24*risiko_gerusan + .21*indeks_deteriorasi
        + .16*paparan_banjir + rng.normal(0, .10, m))
prioritas = skor > 1.10
jembatan = pd.DataFrame({
    "id_aset": [f"J{i:03d}" for i in range(m)],
    "umur_tahun": umur_jembatan, "lalin_berat_harian": lalin_berat,
    "risiko_gerusan": risiko_gerusan.round(3),
    "indeks_deteriorasi": indeks_deteriorasi.round(3),
    "paparan_banjir": paparan_banjir.round(3),
    "prioritas_sintesis": np.where(prioritas, "prioritas", "rutin")})
jembatan.to_csv(out_dir / "data_sintesis_prioritas_jembatan.csv", index=False)
jembatan["prioritas_sintesis"].value_counts(normalize=True)

## 3. Clustering sintesis segmen perkerasan
Kolom `label_pembangkit` hanya untuk diskusi pasca-analisis. Keluarkan kolom itu dan ID dari fitur clustering.

In [4]:
X, z = make_blobs(n_samples=800, centers=4, n_features=4,
                 cluster_std=[0.8, 1.0, 1.2, 0.9], random_state=SEED)
X = (X-X.min(axis=0)) / (X.max(axis=0)-X.min(axis=0))
perkerasan = pd.DataFrame({
    "id_segmen": [f"S{i:04d}" for i in range(len(z))],
    "retak_persen": (X[:, 0]*12).round(2),
    "kedalaman_alur_mm": (X[:, 1]*25).round(2),
    "IRI_m_km": (X[:, 2]*8).round(2),
    "lalu_lintas_tahunan": (X[:, 3]*2.0e6).round(0),
    "label_pembangkit": z})
perkerasan.to_csv(out_dir / "data_sintesis_perkerasan.csv", index=False)
perkerasan.head()

## 4. Sinyal sintesis untuk latihan anomali
Flag simulasi menandai titik yang sengaja digeser. Skor anomali bukan diagnosis kerusakan.

In [5]:
q = 700
waktu = pd.date_range("2024-01-01", periods=q, freq="h")
suhu = 20 + 5*np.sin(np.arange(q)*2*np.pi/24) + rng.normal(0, .5, q)
getaran = .25 + .015*suhu + rng.normal(0, .04, q)
flag = rng.random(q) < .025
getaran[flag] += rng.uniform(.5, 1.4, flag.sum())
sensor = pd.DataFrame({"waktu": waktu, "suhu_C": suhu.round(2),
    "getaran_mm_s": getaran.round(3), "flag_pembangkit": flag})
sensor.to_csv(out_dir / "data_sintesis_sensor.csv", index=False)
sensor.head()

## Audit dan batas penggunaan
Keempat CSV tersimpan dalam `data_bab4`. Pada klasifikasi, ID aset adalah metadata, bukan prediktor. Pada clustering, buang ID dan label pembangkit dari fitur. Tidak satu pun dataset mewakili beton, jembatan, perkerasan, atau sensor nyata.